# Lie Groups for 2D SLAM: $SE(2)$, the Wrap Function, and the Log Map

Manifold and angle-handling preliminaries used by both [pose-graph SLAM](pose_graph_slam.ipynb) and [factor graphs](factor_graph.ipynb). Three sections:

1. **Notation for poses on $SE(2)$** — what $X_i$ really is and how it relates to the triple $(x_i, y_i, \theta_i)$.
2. **The wrap function** — how to subtract angles correctly on the circle $S^1$.
3. **The log map for $SO(2)$ and $SE(2)$** — the formal way to compute orientation residuals on a Lie group.

If you arrive here from another notebook, jump to the section relevant to your derivation; the three sections do not depend on each other.

---


## 1. Notation of $X_i$ and $(x_i, y_i, \theta_i)$


 $(x_i, y_i, \theta_i)$ is a **coordinate representation**,
 $X_i \in SE(2)$ is the **actual pose as a rigid-body transform**,
 and in pose-graph SLAM **nodes live on $SE(2)$**, even if we store them as triples.



**Both are the same pose**, just written in **different representations**.

$$
(x_i, y_i, \theta_i)
$$

→ **minimal coordinates** (vector in $\mathbb{R}^3$)

$$
X_i = \begin{bmatrix} R_i & p_i \\ 0 & 1 \end{bmatrix} \in SE(2)
$$

→ **group (matrix) representation**

They represent **the same physical robot pose in the world frame**.

---

### 1.1 What $X_i$ really is (precise statement)

In pose-graph SLAM:

> Each node $X_i$ is an element of the **Lie group** $SE(2)$

$$
X_i \in SE(2)
$$

The matrix form is the **group element**:

$$
X_i = \begin{bmatrix} R(\theta_i) & p_i \\ 0 & 1 \end{bmatrix}
$$

with

$$
p_i = \begin{bmatrix} x_i \\ y_i \end{bmatrix}, \quad R(\theta_i) = \begin{bmatrix} \cos\theta_i & -\sin\theta_i \\ \sin\theta_i & \cos\theta_i \end{bmatrix}
$$

This matrix **acts on points** and composes via matrix multiplication.

---

### 1.2 Then what is $(x_i, y_i, \theta_i)$?

That triple is **not a group element** by itself.

It is a **coordinate chart** (a parameterization) of $SE(2)$:

$$
\xi_i = \begin{bmatrix} x_i \\ y_i \\ \theta_i \end{bmatrix} \in \mathbb{R}^3
$$

You use it because:

* it is minimal (3 numbers),
* it is convenient for storage and optimization,
* Jacobians are easier to write.

But:

❌ You cannot compose poses by adding these vectors  
❌ You cannot invert poses by negating them

Those operations belong to the group $SE(2)$, not $\mathbb{R}^3$.

---

### 1.3 Relationship between the two

There is a **mapping** from coordinates to group element:

$$
X_i = \mathrm{Exp}(\xi_i)
$$

In practice (for 2D SLAM), this is usually implemented directly as:

$$
(x_i, y_i, \theta_i) \;\longrightarrow\; \begin{bmatrix} R(\theta_i) & p_i \\ 0 & 1 \end{bmatrix}
$$

Strictly speaking, the true Lie-theoretic exponential uses twists, but most 2D SLAM code uses this "direct" embedding because it is exact in $SE(2)$.

---

### 1.4 Why SLAM papers mix the notation

You will often see:

* "Let $X_i = (x_i, y_i, \theta_i)$"
* and later
* "The relative pose is $X_i^{-1} X_j$"

This is **abuse of notation**.

What they mean is:

* **state variables** are stored as $(x,y,\theta)$
* **operations** are done in $SE(2)$

Libraries like **g2o, Ceres, GTSAM** all do this internally.

---

### 1.5 How this shows up in optimization

During optimization you typically have:

* state vector (minimal): $\delta x_i = (\delta x, \delta y, \delta \theta)$
* update rule: $X_i \leftarrow X_i \oplus \delta x_i$

where $\oplus$ means:

$$
X_i \leftarrow X_i \cdot \mathrm{Exp}(\delta x_i)
$$

This is why we **don't add angles directly** during optimization.

---



## 2. Wrap Function

The **wrap** (or **angle normalization**) function maps any real angle to a **principal interval**, usually:

$$
(-\pi,\ \pi]
$$

A standard definition is:

$$
\mathrm{wrap}(\alpha) = (\alpha + \pi)\ \bmod\ (2\pi)\ - \pi
$$

This works for **any real number** $\alpha \in \mathbb{R}$.

Example values:

* $\mathrm{wrap}(3.5) \approx -2.783$
* $\mathrm{wrap}(-3.5) \approx 2.783$
* $\mathrm{wrap}(2\pi + 0.1) = 0.1$
* $\mathrm{wrap}(-4\pi - 0.2) = -0.2$

---

### 2.1 Why angles are special (topology issue)

Angles **do not live in $\mathbb{R}$**.

They live on a **circle** $S^1$:

$$
\theta \equiv \theta + 2\pi k \quad \forall k \in \mathbb{Z}
$$

This means:

* $\theta = -\pi$ and $\theta = \pi$ represent the **same orientation**
* There is **no global linear ordering**

But subtraction assumes a linear space.

---

### 2.2 What goes wrong without wrapping

Suppose the true relative orientation error is tiny:

* predicted: $\hat\theta = +179^\circ$
* measured: $\theta = -179^\circ$

Naive subtraction:

$$
\hat\theta - \theta = 179^\circ - (-179^\circ) = 358^\circ
$$

But geometrically, the orientations differ by only:

$$
2^\circ
$$

This is a **catastrophic error** if you feed it to an optimizer.

Applying wrap:

$$
\mathrm{wrap}(358^\circ) = -2^\circ
$$

Now the residual reflects the **true smallest rotation**.

---

### 2.3 What wrap really computes (geometric meaning)

The wrap function computes the **shortest angular difference** on the circle:

$$
\mathrm{wrap}(\theta_1 - \theta_2) = \operatorname*{argmin}_{\delta \in (-\pi,\pi]} \left| \delta \right| \quad \text{s.t. } \theta_1 = \theta_2 + \delta \ (\text{mod } 2\pi)
$$

In words:

> "What is the smallest rotation that aligns one orientation with the other?"

---

### 2.4 Why SLAM must use wrap in residuals

In pose-graph SLAM, the angular residual is always:

$$
e_\theta = \mathrm{wrap}(\hat\theta_{ij} - \theta_{ij}^{meas})
$$

If you don't wrap:

* residuals jump discontinuously at $\pm \pi$
* the cost function becomes **non-smooth**
* Gauss-Newton / LM can diverge or oscillate
* Hessian and Jacobians become meaningless near $\pm \pi$

Wrap restores **local smoothness**, which is essential for linearization-based solvers.

---

### 2.5 Relation to Lie groups (important intuition)

On $SO(2)$:

* orientation error is not subtraction
* it is **group difference**

$$
R_{\text{err}} = R(\hat\theta)^T R(\theta)
$$

The angle of $R_{\text{err}}$ is **automatically wrapped** into $(-\pi,\pi]$.

Using wrap is a **cheap coordinate-level substitute** for doing the full group computation.

---

### 2.6 Why wrap is NOT needed for translation

* $x, y \in \mathbb{R}$ → linear space
* subtraction is well-defined globally
* no periodicity

Only angles have this issue.

---

### 2.7 Typical wrap implementations

Mathematically correct and copy-paste safe:

$$
\mathrm{wrap}(\alpha) = (\alpha + \pi) \bmod (2\pi) - \pi
$$

Alternative equivalent form:

$$
\mathrm{wrap}(\alpha) = \operatorname{atan2}(\sin\alpha,\ \cos\alpha)
$$

Both map to $(-\pi,\pi]$.

---

### 2.8 When wrap is *not* enough (advanced note)

If angular error exceeds $\pi$:

* linearization is invalid
* optimizer may converge to the wrong local minimum

That's why:

* good initialization matters
* loop-closure verification is critical
* some systems re-linearize on manifolds explicitly (SE(2), SE(3))

---

### 2.9 One-sentence takeaway

> **Wrap is needed because angles live on a circle, not a line — it ensures the angular residual represents the shortest physical rotation and keeps SLAM optimization smooth and correct.**



## 3. The Log Map for $SO(2)$

For a 2D rotation matrix:

$$
R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}
$$

The **log map** is:

$$
\log: SO(2) \to \mathbb{R}
$$

Given a rotation matrix $R$, we recover its angle $\theta$:

$$
\theta = \operatorname{atan2}(R_{21}, R_{11})
$$

where $R_{21} = \sin\theta$, $R_{11} = \cos\theta$.

The key: the $\operatorname{atan2}$ function returns values in $(-\pi,\pi]$, so $\log(R)$ is **automatically wrapped**.

---

### 3.1 Composition and error on $SO(2)$

True relative pose computation without coordinates:

Let $R_i, R_j \in SO(2)$ be the rotation matrices.

The **predicted relative rotation** from $i$ to $j$ is:

$$
\hat{R}_{ij} = R_i^T R_j
$$

The **measured relative rotation** is $R_{ij}^{\text{meas}} = R(\theta_{ij}^{\text{meas}})$.

The **error rotation** is:

$$
R_{\text{err}} = (R_{ij}^{\text{meas}})^T \hat{R}_{ij}
$$

The angular error is:

$$
e_\theta = \log(R_{\text{err}})
$$

This is **mathematically equivalent** to:

$$
e_\theta = \mathrm{wrap}(\hat\theta_{ij} - \theta_{ij}^{\text{meas}})
$$

---

### 3.2 Proof of equivalence

Let's prove the equivalence:

1. $\hat\theta_{ij} = \theta_j - \theta_i$ (mod $2\pi$)
2. Compute $R_{\text{err}} = R(-\theta_{ij}^{\text{meas}}) R(\theta_j - \theta_i) = R(\theta_j - \theta_i - \theta_{ij}^{\text{meas}})$
3. $\log(R_{\text{err}}) = \theta_j - \theta_i - \theta_{ij}^{\text{meas}}$ (wrapped to $(-\pi,\pi]$)
4. This equals $\mathrm{wrap}(\hat\theta_{ij} - \theta_{ij}^{\text{meas}})$

✅ The wrap function is essentially the log map in coordinate form.

---

### 3.3 Extension to $SE(2)$

For full 2D poses $(x, y, \theta) \in SE(2)$, the error is:

$$
e_{ij} = \log\left( T_{ij}^{\text{meas}, -1} \cdot T_i^{-1} T_j \right)
$$

where $T_i = \begin{bmatrix} R(\theta_i) & p_i \\ 0 & 1 \end{bmatrix} \in SE(2)$.

The log map for $SE(2)$ returns a 3D vector:

$$
\log: SE(2) \to \mathbb{R}^3
$$

First two components: translation in the body frame  
Third component: rotation angle (wrapped)

---

### 3.4 What happens in $SE(3)$

In 3D, rotations are in $SO(3)$. The log map is more involved:

For $R \in SO(3)$, the angle-axis representation:

$$
\log(R) = \frac{\phi}{2\sin\phi}(R - R^T)^\vee
$$

where $\phi = \arccos\left(\frac{\operatorname{tr}(R) - 1}{2}\right)$.

The result is a 3D vector $\omega \in \mathbb{R}^3$ with $|\omega| \leq \pi$.

This automatically **wraps** large rotations to the shortest geodesic.

---

### 3.5 Why this matters for optimization

When we linearize the cost function:

$$
e_{ij}(X + \delta) \approx e_{ij}(X) + J_{ij}\delta
$$

The Jacobian $J_{ij}$ relates **state perturbations** $\delta \in \mathbb{R}^3$ to **error changes**.

If we use naive angle subtraction without wrap, the Jacobian is wrong when errors are near $\pm\pi$:

**With wrap** (log map):
$$
\frac{\partial e_\theta}{\partial \theta_i} = -1
$$

**Without wrap** (at discontinuity):
$$
\frac{\partial e_\theta}{\partial \theta_i} \text{ can be } \pm 2\pi
$$

This wrong Jacobian breaks Gauss-Newton convergence.

---

### 3.6 Implementation in practice

In C++ libraries like g2o, Ceres, GTSAM:

```cpp
// The "right" way using SE(2) operations
SE2 Ti(pose_i), Tj(pose_j), Tij_meas(measurement);
SE2 error = Tij_meas.inverse() * Ti.inverse() * Tj;
Vector3 e = error.log();  // Automatically wrapped

// Equivalent to:
Vector3 e;
e.head<2>() = R_i.transpose() * (p_j - p_i) - t_ij_meas;
e(2) = wrap(theta_j - theta_i - theta_ij_meas);
```

---

### 3.7 One-liner intuition

> **wrap = log map in coordinate form**

The wrap function isn't a hack — it's the coordinate representation of the mathematically correct **logarithmic map** from the Lie group to its tangent space.

---

### 3.8 Takeaway for SLAM implementation

1. **Always use wrap for angle residuals**
2. It's equivalent to using the log map
3. It ensures smooth, correct optimization
4. For 3D, use proper $SO(3)$/quaternion operations
5. Libraries handle this internally; understand why

---
